In [1]:
import sys

sys.path.append("..")


from src.plotting.feature_plotting import plot_graph
import matplotlib.pyplot as plt
import numpy as np
import torch

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
from src.data_preparation import load_numpy_files

signal_prefix = f"{DATA_DIR}/sig"
background_prefix = f"{DATA_DIR}/bg"
signal_only_prefix = f"{DATA_DIR}/sig_only"


In [2]:
import src.torch.pre_processing.graph_batching as graph_batching
from importlib import reload
reload(graph_batching)

<module 'src.torch.pre_processing.graph_batching' from '/Users/simi/mu3e_trigger/notebooks_supervised/../src/torch/pre_processing/graph_batching.py'>

In [ ]:
train_dataset, val_dataset, test_dataset = graph_batching.create_dataset(
    background_prefix,
    has_layer_feature=True,
    n_events=100000,
    split=(0.7, 0.15, 0.15),
    type="hetero",
    mppc_timing_cutoff=1)


In [4]:
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [5]:
from torch_geometric.data import HeteroData
from src.torch.model.components import get_mlp
class EdgeClassifier(torch.nn.Module):
    def __init__(self, node_dims, edge_types, num_layers, hidden_dim = 64):
        super(EdgeClassifier, self).__init__()
        from torch_geometric.nn import HeteroConv, SAGEConv, Linear
        self.input_embeddings = torch.nn.ModuleDict()
        self.convs = torch.nn.ModuleList()
        for node_type, node_dim in node_dims.items():
            self.input_embeddings[node_type] = get_mlp(node_dim, hidden_dim, 2)
        for i in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv((-1, -1), hidden_dim)
                for edge_type in edge_types
            }, aggr='max')
            self.convs.append(conv)
        self.lin = get_mlp(2 * hidden_dim, 1, 4)

    def forward(self, hetero_graph: HeteroData):
        x_dict = hetero_graph.x_dict
        edge_index_dict = hetero_graph.edge_index_dict

        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: x.relu() for key, x in x_dict.items()}

        edge_preds = {}
        for edge_type, edge_index in hetero_graph.edge_index_dict.items():
            src_type, _, dst_type = edge_type
            src_x = x_dict[src_type]
            dst_x = x_dict[dst_type]
            edge_feat = torch.cat([src_x[edge_index[0]], dst_x[edge_index[1]]], dim=-1)
            edge_pred = self.lin(edge_feat).view(-1)
            edge_preds[edge_type] = edge_pred
        return edge_preds

In [6]:
edge_classifier = EdgeClassifier(
    node_dims=train_dataset.get_node_dims(),
    edge_types=train_dataset.get_edge_types(),
    num_layers=5,
    hidden_dim=32
)
#print(f"Number of parameters: {sum(p.numel() for p in edge_classifier.parameters() if p.requires_grad)}")

In [7]:
from src.torch.training import FocalLoss, HeteroLossWrapper
from sklearn.metrics import roc_auc_score
criterion = HeteroLossWrapper(FocalLoss())
optimizer = torch.optim.Adam(edge_classifier.parameters(), lr=0.001)

from tqdm import tqdm
num_epochs = 20
for epoch in range(num_epochs):
    edge_classifier.train()
    total_loss = 0
    for hetero_graph in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()
        out = edge_classifier(hetero_graph)
        # Assuming labels are stored in hetero_graph.y
        loss = criterion(out, hetero_graph.edge_labels_dict)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1/50: 100%|██████████| 636/636 [02:17<00:00,  4.63it/s]


Epoch 1, Loss: 0.1586


Epoch 2/50: 100%|██████████| 636/636 [02:48<00:00,  3.78it/s]


Epoch 2, Loss: 0.1546


Epoch 3/50: 100%|██████████| 636/636 [02:40<00:00,  3.96it/s]


...

Epoch 22, Loss: 0.1360


Epoch 23/50: 100%|██████████| 636/636 [02:43<00:00,  3.88it/s]


Epoch 23, Loss: 0.1366


Epoch 24/50:  13%|█▎        | 80/636 [00:19<02:16,  4.08it/s]


KeyboardInterrupt: 

In [8]:
predictions = {}
labels = {}
edge_classifier.eval()
with torch.no_grad():
    for hetero_graph in tqdm(test_loader, desc="Evaluating"):
        out = edge_classifier(hetero_graph)
        for edge_type in out:
            if edge_type not in predictions:
                predictions[edge_type] = []
                labels[edge_type] = []
            predictions[edge_type].append(torch.sigmoid(out[edge_type].cpu()))
            labels[edge_type].append(hetero_graph.edge_labels_dict[edge_type].cpu())
    for edge_type in predictions:
        predictions[edge_type] = torch.cat(predictions[edge_type])
        labels[edge_type] = torch.cat(labels[edge_type])
        auc = roc_auc_score(labels[edge_type].numpy(), predictions[edge_type].numpy())
        print(f"Edge type: {edge_type}, AUC: {auc:.4f}")

Evaluating: 100%|██████████| 136/136 [00:14<00:00,  9.53it/s]


Edge type: ('pixel', 'to', 'pixel'), AUC: 0.8215
Edge type: ('pixel', 'to', 'mppc'), AUC: 0.7727
Edge type: ('mppc', 'to', 'pixel'), AUC: 0.7724
Edge type: ('mppc', 'to', 'mppc'), AUC: 0.5669
